In [ ]:
import os
os.environ["AMDGPU_TARGETS"] = "gfx1032"
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
import time
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
from dotenv import load_dotenv
import math 
import time
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler, Dataset, random_split
import cv2
import torch.nn as nn
import torch.optim as optim
from EAST import EAST
import torch.nn.functional as F


In [ ]:
#controls

rboxOn = True


In [ ]:
class imgDataset(Dataset):
    def __init__(self, path_imgs, path_labelAndcoords, box_shrink = 0.85):
        super().__init__()
        self.path_imgs = path_imgs
        self.path_labelAndCoords = path_labelAndcoords
        self.box_shrink = box_shrink

    def __len__(self):
        return len(self.path_imgs)
    
    def __getitem__(self, index):

        #img to tensor
        path_img = self.path_imgs[index]
        img = Image.open(path_img)
        transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.ToTensor(),

        ])
        img = transform(img)

        labelsForImg = []
        coordsForImg = []
        textAndCoord = self.path_labelAndCoords[index]

        #read the text file and parse the labels and coordinates
        with open(textAndCoord, 'r', encoding='utf-8-sig') as file: 
            line = file.readline()
            while line:
                line = line.strip()
                if line:
                    labelAndcoord = line.split(',')
                    labelsForImg.append([i for i in labelAndcoord[8]])
                    coords = [int(float(f) / 1) for f in labelAndcoord[:8]]
                    coordsForImg.append(coords)       
                line = file.readline() 
        # make coordmap for the image
        TrueMap = torch.zeros(1,img.shape[1], img.shape[2])
        if rboxOn:
            corners = torch.zeros(4,img.shape[1], img.shape[2])
        else:
            corners = torch.zeros(8,img.shape[1], img.shape[2])

        for box in coordsForImg:

            #get center

            cx = (box[0] + box[2] + box[4] + box[6]) / 4
            cy = (box[1] + box[3] + box[5] + box[7]) / 4

            shrunkn_box = []
            for i in range(0,8,2):
                x_point = box[i]
                y_point = box[i+1]
                x_new = int(cx + self.box_shrink*(x_point - cx))
                y_new = int(cy + self.box_shrink*(y_point - cy))
                shrunkn_box.append(x_new)
                shrunkn_box.append(y_new)

            # parsing coords
            # gets egdes
            edges = [
                (shrunkn_box[0:2], shrunkn_box[2:4]), # (933,255) → (954,255)  top edge
                (shrunkn_box[2:4], shrunkn_box[4:6]),  # (954,255) → (956,277)  right edge
                (shrunkn_box[4:6], shrunkn_box[6:8]), # (956,277) → (936,277)  bottom edge
                (shrunkn_box[6:8], shrunkn_box[0:2])  # (936,277) → (933,255)  left edge
            ]

            #bounding box
            #get coreners of box not boundning
            corners_xy = [(shrunkn_box[j], shrunkn_box[j+1]) for j in range(0, 8, 2)]
            min_x = min(shrunkn_box[0], shrunkn_box[2], shrunkn_box[4], shrunkn_box[6])
            max_x = max(shrunkn_box[0], shrunkn_box[2], shrunkn_box[4], shrunkn_box[6])
            min_y = min(shrunkn_box[1], shrunkn_box[3], shrunkn_box[5], shrunkn_box[7])
            max_y = max(shrunkn_box[1], shrunkn_box[3], shrunkn_box[5], shrunkn_box[7])

            for x in range(min_x, max_x + 1):
                for y in range(min_y, max_y + 1):
                    cnt = 0
                    for edge in edges:
                        (x1,y1), (x2,y2) = edge
                        if (y < y1) != (y < y2) and y2 != y1 and x < x1 + (y - y1) / (y2 - y1) * (x2 - x1):
                            cnt += 1
                    if cnt % 2 == 1:
                        TrueMap[0][y][x] = 1
                    
                    if TrueMap[0][y][x] == 1 and False == rboxOn:  # only for pixels inside the box
                        for k, (cxt, cyt) in enumerate(corners_xy):
                            corners[k*2][y][x]   = cxt - x  # ∆x to corner k
                            corners[k*2+1][y][x] = cyt - y  # ∆y to corner k
                            # add quad version
                    else:
                        # rbox  channels
                        for k, edge in enumerate(edges):
                            if TrueMap[0][y][x] == 1:
                                (x1,y1),(x2,y2) = edge
                                top = abs(((x2-x1)*(y1-y))-((x1-x)*(y2-y1)))
                                bottom = math.sqrt((x2-x1)**2+(y2-y1)**2)
                                if top == 0 or bottom == 0:
                                    corners[k][y][x] = 0
                                else:
                                    corners[k][y][x] = top/bottom
        
        res = 512
        img=img.unsqueeze(0)
        imga = F.interpolate(img, size=(res, res) , mode='bilinear', align_corners=False)
        imga=imga.squeeze(0)

        TrueMap=TrueMap.unsqueeze(0)
        TrueMapa = F.interpolate(TrueMap, size=(res, res) , mode='bilinear', align_corners=False)
        TrueMapa=TrueMapa.squeeze(0)

        corners=corners.unsqueeze(0)
        cornersa = F.interpolate(corners, size=(res, res) , mode='bilinear', align_corners=False)
        cornersa=cornersa.squeeze(0)

        return [imga, TrueMapa, cornersa]

In [ ]:
def test_img_dataset():
    load_dotenv()
    directory_train = os.getenv('Directory_train')
    print(directory_train)
    directory_train_textAndCoords = os.getenv('directory_train_textAndCoords')
    print(directory_train_textAndCoords)
    # Generate paths for training images and labels
    train_img_paths = sorted([os.path.join(directory_train, f) for f in os.listdir(directory_train)])
    print(train_img_paths[0])
    train_label_paths = sorted([os.path.join(directory_train_textAndCoords, f) for f in os.listdir(directory_train_textAndCoords)])
    print(train_label_paths[0])
    #making objs
    train_dataset = [train_img_paths, train_label_paths]
    train_dataset = imgDataset(train_img_paths, train_label_paths)
    img, TrueMap, corners = train_dataset[5]

    print(f"shape: {img.shape}", type(img)) # Should be [1, 360, 360] since it's grayscale
    print(f"TrueMap: {TrueMap.shape}",type(TrueMap))
    print(f"corners:{corners.shape}",type(corners))
    return img, TrueMap, corners

if 1==1:
    img, TrueMap, corners = test_img_dataset()

In [ ]:
# loss funtions
# ls
def balanced_cross_entropy_loss(preds, targets, epsilon=1e-3) -> torch.Tensor:
    beta = 1 - torch.mean(targets.float())
    print(f"beta={beta:.4f}")
    preds = torch.clamp(preds, epsilon, 1.0 - epsilon)
    return ((-beta * targets * torch.log(preds)) - 
            (1-beta)*(1-targets)*(torch.log(1-preds))).mean()

#lg this is what is breaking
def quad_loss(preds, targets, TrueMap, epsilon=1e-3) -> torch.Tensor :
    mask = TrueMap
    loss = torch.nn.functional.smooth_l1_loss(preds * mask, targets * mask, reduction='sum')
    normalizer =(mask.sum() + epsilon) * 8
    return loss / ( normalizer)

#four if i do geo as4 #
def rbox(pred: torch.Tensor, gt: torch.Tensor, TrueMap: torch.Tensor, smooth: float = 1e-3) -> torch.Tensor:

    mask = (TrueMap != 0)

    inter_h = (torch.min(pred[:, 0], gt[:, 0]) + torch.min(pred[:, 2], gt[:, 2]))
    inter_w = (torch.min(pred[:, 3], gt[:, 3]) + torch.min(pred[:, 1], gt[:, 1]))
    inter   = inter_h * inter_w

    pred_area = (pred[:, 0] + pred[:, 2]) * (pred[:, 1] + pred[:, 3])
    gt_area   = (gt[:, 0]   + gt[:, 2])   * (gt[:, 1]   + gt[:, 3])
    union     = (pred_area + gt_area - inter + smooth)

    iou  = ((inter + smooth) / union)
    loss = -torch.log(iou.clamp(min=1e-6))
    loss = loss* mask

    if loss.sum() == 0:
        return (pred * 0).sum()
    return loss.sum() / mask.sum()

In [ ]:
#test loss
def test_loss_funtions(img,TrueMap,corners):
    torch.cuda.empty_cache()
    model = EAST(color_channel=1,scale_factor=4)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    img = img.unsqueeze(0)
    img = img.to(device)
    TrueMap = TrueMap.unsqueeze(0)
    TrueMap = TrueMap.to(device)
    corners = corners.unsqueeze(0)
    corners = corners.to(device)

    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    optimizer.zero_grad()

    outputs = model(img)

    score_map , geo_map, quad_geo_map = outputs
    print("base output: ----------")
    print("score map:",score_map.max())
    print("score map:",score_map.min())
    print("geo_map:",geo_map.max())
    print("geo_map:",geo_map.min())
    print("quad_geo_map:",quad_geo_map.max())
    print("quad_geo_map:",quad_geo_map.min())

    a = balanced_cross_entropy_loss(preds=score_map,targets=TrueMap)
    print("balanced_cross_entropy_loss: ", a)
    c = rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
    print("rbox: ", c)
    d = a + c
    print("comp", d)

    print(d)
test_loss_funtions(img,TrueMap,corners)

In [ ]:
#test img and coords
def visualize_rbox_edges(img, TrueMap, corners):
    import cv2
    import numpy as n
    img_np     = img.squeeze().cpu().numpy()          # (H, W)
    score_np   = TrueMap.squeeze().cpu().numpy()      # (H, W)
    corners_np = corners.cpu().numpy()                # (4, H, W)

    H, W = score_np.shape
    edge_names = ["Top Edge", "Right Edge", "Bottom Edge", "Left Edge"]

    panels = []

    for k in range(4):
        canvas = (img_np * 255).astype(np.uint8)
        canvas = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)

        dist_map = corners_np[k]                      # (H, W) raw distances

        # only care about pixels inside the score map
        mask = score_np == 1
        if mask.sum() == 0:
            panels.append(canvas)
            continue

        dist_inside = dist_map[mask]
        d_min = dist_inside.min()
        d_max = dist_inside.max()
        d_range = d_max - d_min if d_max != d_min else 1.0

        # brightness = inverse distance (closer → brighter)
        heat = np.zeros((H, W), dtype=np.float32)
        heat[mask] = 1.0 - (dist_map[mask] - d_min) / d_range   # [0,1], 1=closestloss_geo_map

        # map to colormap (INFERNO: dark=far, bright=close)
        heat_u8  = (heat * 255).astype(np.uint8)
        colored  = cv2.applyColorMap(heat_u8, cv2.COLORMAP_INFERNO)

        # blend only inside-mask pixels onto the grayscale canvas
        mask_3ch = np.stack([mask]*3, axis=-1)
        blended  = np.where(mask_3ch, colored, canvas)

        # label
        cv2.putText(blended, edge_names[k], (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2)

        panels.append(blended)

    # stack 2x2 grid
    top    = np.hstack([panels[0], panels[1]])
    bottom = np.hstack([panels[2], panels[3]])
    grid   = np.vstack([top, bottom])

    cv2.imshow("RBOX Edge Distance Heatmaps", grid)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
if 1==0:
    visualize_rbox_edges(img, TrueMap, corners)


In [ ]:
def train_cycle(model, dataset_loaded, device, optimizer):
    start = time.time()
    running_loss = 0.0
    count = 0
    accumulation_steps = 4
    #img, TrueMap, corners
    for i,(imgs, TrueMap, corners) in enumerate(dataset_loaded):

        count += len(TrueMap)

        imgs = imgs.to(device)
        TrueMap = TrueMap.to(device)
        corners = corners.to(device)
        score_map, geo_map, quad_geo_map = model(imgs)
        
        loss_score_map = balanced_cross_entropy_loss(preds = score_map, targets = TrueMap)
        loss_geo_map = rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
        total_loss = loss_score_map + (1.0 * loss_geo_map) 

        scaled_loss = total_loss / accumulation_steps
        print(f"bce={loss_score_map.item():.4f}  rbox={loss_geo_map.item():.4f}")
        scaled_loss.backward()
        running_loss += total_loss.item()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
    if (i + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = running_loss / len(dataset_loaded)
    print(f"Time taken: {time.time() - start:.2f} seconds")
    print(f"Avg Loss: {avg_loss:.4f}")
    return avg_loss

def val_cycle(model, dataset_loaded, device, optimizer):

    model.eval()
    start = time.time()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    total_iou = 0 
    accumulation_steps = 4
    
    with torch.no_grad():
        for imgs, TrueMap, corners in dataset_loaded:

            imgs = imgs.to(device)
            TrueMap = TrueMap.to(device)
            corners = corners.to(device)

            score_map, geo_map, quad_geo_map = model(imgs)

            loss_score_map = balanced_cross_entropy_loss(preds = score_map, targets = TrueMap)
            loss_geo_map = rbox(pred = quad_geo_map, gt = corners, TrueMap = TrueMap)
            total_loss = loss_score_map + (1.0 * loss_geo_map)
            running_loss += total_loss.item()

            preds_binary = score_map
            intersection = torch.logical_and(preds_binary, TrueMap).sum().item()
            union = torch.logical_or(preds_binary, TrueMap).sum().item()
            iou = intersection / union if union > 0 else 0.0
            total_iou += iou


    avg_loss = running_loss / len(dataset_loaded)
    correct_avg = total_iou / len(dataset_loaded)

    print(f"Time taken: {time.time() - start:.2f} seconds")
    print(f"Avg Loss: {avg_loss:.4f}")
    return correct_avg, avg_loss

In [ ]:
def train_model(model, loader_train, loader_val, scheduler, optimizer, cycles):
    model.train()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    model.to(device)

    best_val_acc = 100
    best_model_path = "east_model.pth"

    for cycle in range(cycles):
        print(f"cycle: {cycle+1}/{cycles}")

        #training within the cycle
        model.train()
        train_loss = train_cycle(model, dataset_loaded = loader_train, device = device,optimizer = optimizer)
        
        model.eval()
        val_acc, val_loss = val_cycle(model, dataset_loaded = loader_val, device = device,optimizer = optimizer)
        scheduler.step()
        
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}")
        if val_loss < best_val_acc:
            best_val_acc = val_loss
            print(f"model beats old with Val Acc: {best_val_acc:.2f}%, saving model.")
            torch.save(model.state_dict(), "east_model.pth")

    return best_model_path


In [ ]:
#loads in paths
load_dotenv()
directory_train = os.getenv('Directory_train')
directory_train_textAndCoords = os.getenv('directory_train_textAndCoords')

In [ ]:
# Define path_imgs and path_labels
path_imgs = sorted(os.listdir(directory_train))
path_labels = sorted(os.listdir(directory_train_textAndCoords))

# Generate paths for training images and labels
train_img_paths = [os.path.join(directory_train, f) for f in path_imgs]
train_label_paths = [os.path.join(directory_train_textAndCoords, f) for f in path_labels]

In [ ]:
#making objs
train_dataset = [train_img_paths, train_label_paths]
train_dataset = imgDataset(train_img_paths, train_label_paths)
img, TrueMap, corners = train_dataset[0]

print(f"Image shape: {img.shape}", type(img)) # Should be [1, 360, 360] since it's grayscale
print(f"First TrueMap: {TrueMap.shape}",type(TrueMap))
print(f"First corners: {corners.shape}",type(corners))

#batch = custom_collate(train_dataset)


In [ ]:
#spltining the dataset into training and validation sets
val_split = 0.2
training_size = int((1 - val_split) * len(train_dataset))
val_size = len(train_dataset) - training_size
training_dataset, val_dataset = random_split(train_dataset, [training_size, val_size])
loader_train = DataLoader(training_dataset, batch_size=6, shuffle=True,num_workers=12, pin_memory=True)
loader_val = DataLoader(val_dataset, batch_size=3, shuffle=True,num_workers=8, pin_memory=True)

In [ ]:
#loading model
model = EAST(color_channel=1, scale_factor=4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
model.Initialize_weights()
model.to(device)


In [ ]:
# Loading model if it exists
load_path = os.getenv('Load_model')

if load_path and os.path.isfile(load_path):
    model.load_state_dict(torch.load(load_path))
    print(f"Model loaded successfully from: {load_path}")
else:
    print("Model not loaded, starting from scratch")

In [ ]:
#defualt model otipions
optimizer = optim.Adam(model.parameters(), lr = 0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
cycles = 5

In [ ]:
train_model(model = model, loader_train = loader_train, loader_val = loader_val,scheduler=scheduler, optimizer = optimizer, cycles = cycles)

In [ ]:
model_save_path = os.getenv('Model_save_path', 'east_model.pth')
torch.save(model.state_dict(), model_save_path)